## Setup

In [14]:
import os
import random
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from cmdstanpy import CmdStanModel
import cmdstanpy
import numpy as np
from scipy.stats import invgamma
from scipy.stats import norm
from scipy.special import softmax
from gptools.stan import get_include

import pandas as pd
from glob import glob
from nteprsm import utils 
from cmdstanpy import stanfit
from settings import ROOT_DIR
import plotly.express as px
import utils as notebook_utils
# use customize plotly template
notebook_utils.set_custom_template()

import pickle
import arviz as az

import plotly.subplots as sp
from matplotlib import colors as mcolors

In [10]:
df = pd.read_csv(ROOT_DIR/'data/raw/quality_nj2.csv')
df = df.assign(
    #entry_name_code=pd.Categorical(df["entry_name"]).codes,
    PLT_ID_CODE=pd.Categorical(df["PLT_ID"]).codes,
    #rater_code=pd.Categorical(df["rater"]).codes,
    #rating_event_code=pd.Categorical(df["rating_event"]).codes,
)

# Define grid dimensions (num_rows and num_cols should match your grid)
num_rows = int(df['ROW'].max())  # Maximum row index
num_cols = int(df['COL'].max())  # Maximum column index

# comparison analysis
with open('old_model_fit2', 'rb') as file:
    fit_old_model = pickle.load(file)
with open('annual_seasonality_nj2.pkl', 'rb') as file:
    fit = pickle.load(file)

FileNotFoundError: [Errno 2] No such file or directory: 'old_model_fit2'

In [ ]:
NTEP_COLOR_SCALE = ["#8c510a","#bf812d","#dfc27d","#f6e8c3","#c7e9c0","#74c476","#41ab5d","#238b45","#00441b"]

In [ ]:
def rsm_probability(y, theta, thresholds):
    """
    Calculates the probability of a given class label in the model.

    Args:
    y (int): The class label for which the probability is calculated.
    theta (np.ndarray): An array of model parameters.
    tau (np.ndarry): The threshold parameters for the model.

    Returns:
    float: The probability of the given class label.

    """
    unsummed = np.concatenate(([0], theta - thresholds))
    probs = softmax(np.cumsum(unsummed))
    return probs[y]

def rsm_probability_vector(theta, thresholds):
    """
    Calculates the probability of set of class labels in given model

    Args:
    theta (np.ndarray): An array of model parameters.
    tau (np.ndarry): The threshold parameters for the model.

    Returns:
    np.ndarray: Array probability of given class label.
    """
    unsummed = np.concatenate(([0], theta - thresholds))
    probs = softmax(np.cumsum(unsummed))
    return probs
    
def plot_rater_characteristic_curve(
        betas,
        min_theta=-6,
        max_theta=6,
        resolution=500,
        colors=px.colors.diverging.Spectral,
        dimensions=None,
    ) -> go.Figure:
        """
        Plot the characteristic curves for raters based on the fitted Stan model.

        Args:
            rater_id (int, optional): The rater ID to plot. If None, all raters
                will be plotted. Defaults to None.
            dimensions (tuple, optional): Dimensions of the plot as (width, height).

        Returns:
            A Plotly figure object containing the plotted characteristic curves.
        """
        betas_with_bounds = np.concatenate(([min_theta], betas, [max_theta]))
        x = np.linspace(min_theta, max_theta, int((max_theta - min_theta) * resolution))
        num_categories = len(betas) + 1
        fig = go.Figure()
        fig.update_layout(
            template="ggplot2",
            xaxis_title="Turf Quality on Latent Scale",
            yaxis_title="Probability",
            legend=dict(x=1.02, y=1),
        )
        for i in range(num_categories):
            fig.add_trace(
                go.Scatter(
                    x=x,
                    y=[rsm_probability(i, theta, betas) for theta in x],
                    line=dict(width=2, color=colors[i]),
                    name=str(i + 1),
                )
            )
            fig.add_shape(
                type="rect",
                x0=betas_with_bounds[i],
                x1=betas_with_bounds[i + 1],
                y0=1.02,
                y1=1.1,
                fillcolor=colors[i],
            )
            if i != num_categories - 1:
                fig.add_shape(
                    type="line",
                    x0=betas[i],
                    x1=betas[i],
                    y0=0,
                    y1=1,
                    line=dict(color=colors[i], dash="dot"),
                )
        if dimensions:
            fig.update_layout(width=dimensions[0], height=dimensions[1])
        return fig

def hex_to_rgba(hex_color, alpha=0.2):
    """Convert a HEX color (e.g., '#1f77b4') to an RGBA string with transparency."""
    rgb = mcolors.hex2color(hex_color)  # Convert hex to RGB (normalized 0-1)
    return f'rgba({int(rgb[0]*255)}, {int(rgb[1]*255)}, {int(rgb[2]*255)}, {alpha})'

def map_name2code(datahandler, column_name, code_column_name, invert=False):
    """
    Retrieves a dictionary mapping names to codes from specified columns.

    Args:
        column_name(str): The name of the column containing the names(
        e.g., 'ENTRY_NAME', 'RATER').
        code_column_name(str): The name of the column containing the codes
        (e.g., 'ENTRY_NAME_CODE', 'RATER_CODE').
        invert(bool): If True, returns a dictionary mapping codes to names.

    Returns:
        dict: Depending on 'invert', returns either a dict of {name: code}
        or {code: name}.
    """
    name2code = dict(datahandler.model_data.groupby(column_name)[code_column_name].first())


## Spatial effect comparison

In [ ]:
# Initialize heatmaps with NaNs to indicate missing values
heatmap = np.full((num_rows, num_cols), np.nan)
heatmap_old_model = np.full((num_rows, num_cols), np.nan)

grouped = df.groupby('PLT_ID_CODE')[['ROW', 'COL']].mean()
rows = grouped.to_dict()['ROW']
cols = grouped.to_dict()['COL']

plot_effect_means = fit.plot_effect.mean(axis = 0)
plot_effect_means_old_model = fit_old_model.plot.mean(axis = 0)

for i in df['PLT_ID_CODE'].unique():
    row = int(rows[i]) - 1
    col = int(cols[i]) - 1
    heatmap[row, col] = plot_effect_means[i]
    heatmap_old_model[row, col] = plot_effect_means_old_model[i]

In [ ]:
# Create a figure with two subplots
fig, axes = plt.subplots(1, 2, figsize=(12, 6), sharey=True)

# Define colormap to handle NaN values properly (white color for NaNs)
cmap = plt.cm.viridis.copy()
cmap.set_bad(color='white')  # NaNs appear as white

# Define different normalization scales for each heatmap
norm_gp = mcolors.Normalize(vmin=np.nanmin(heatmap), vmax=np.nanmax(heatmap))  # Scale for GP Model
norm_old = mcolors.Normalize(vmin=np.nanmin(heatmap_old_model), vmax=np.nanmax(heatmap_old_model))  # Scale for Old Model

# Plot first heatmap (GP Model)
im1 = axes[0].imshow(heatmap, aspect='auto', cmap=cmap, norm=norm_gp)
axes[0].set_title("Spatial Effect GP (Annual Seasonality)")

# Plot second heatmap (Old Model) with a different scale
im2 = axes[1].imshow(heatmap_old_model, aspect='auto', cmap=cmap, norm=norm_old)
axes[1].set_title("Spatial Effect Old Model")

# Compute tick positions at the center of each grid cell
x_ticks = np.arange(num_cols)  # Centered ticks for columns
y_ticks = np.arange(num_rows)  # Centered ticks for rows
x_labels = np.arange(1, num_cols + 1)  # 1-based labels for columns
y_labels = np.arange(1, num_rows + 1)  # 1-based labels for rows

# Set x and y ticks for both plots
for ax in axes:
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(x_labels)
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)

# Ensure y-axis starts from 1 at the bottom
axes[0].invert_yaxis()  # Matplotlib starts from top, so we invert

# Add colorbars with independent scales
cbar1 = fig.colorbar(im1, ax=axes[0], orientation="vertical", fraction=0.05)
cbar1.set_label("GP Model Effect")

cbar2 = fig.colorbar(im2, ax=axes[1], orientation="vertical", fraction=0.05)
cbar2.set_label("Old Model Effect")

# Add axis labels
axes[0].set_xlabel("Plot Column")
axes[1].set_xlabel("Plot Column")
axes[0].set_ylabel("Plot Row")  # Only need y-label for the first plot

# Show the plot
plt.tight_layout()
plt.show()

In [ ]:
# Compute the difference between the two heatmaps
heatmap_diff = heatmap - heatmap_old_model  # Element-wise difference

# Define normalization scale for the difference heatmap
norm_diff = mcolors.Normalize(vmin=np.nanmin(heatmap_diff), vmax=np.nanmax(heatmap_diff))

# Create a figure with one subplot for the difference heatmap
fig, ax = plt.subplots(1, 1, figsize=(8, 6))

# Plot the difference heatmap
im_diff = ax.imshow(heatmap_diff, aspect='auto', cmap=plt.cm.coolwarm, norm=norm_diff)
ax.set_title("Mean difference in Spatial Effects (GP - Old Model)")

# Compute tick positions at the center of each grid cell
x_ticks = np.arange(num_cols)  # Centered ticks for columns
y_ticks = np.arange(num_rows)  # Centered ticks for rows
x_labels = np.arange(1, num_cols + 1)  # 1-based labels for columns
y_labels = np.arange(1, num_rows + 1)  # 1-based labels for rows

# Set x and y ticks
ax.set_xticks(x_ticks)
ax.set_xticklabels(x_labels)
ax.set_yticks(y_ticks)
ax.set_yticklabels(y_labels)

# Ensure y-axis starts from 1 at the bottom
ax.invert_yaxis()

# Add a colorbar for the difference heatmap
cbar_diff = fig.colorbar(im_diff, ax=ax, orientation="vertical", fraction=0.05)
cbar_diff.set_label("Mean difference in Effects (New Model - Old Model)")

# Add axis labels
ax.set_xlabel("Plot Column")
ax.set_ylabel("Plot Row")

# Show the plot
plt.tight_layout()
plt.show()

## ELPD comparison

In [11]:
import arviz as az
azdata = az.from_cmdstanpy(fit)
azdata_old = az.from_cmdstanpy(fit_old_model)

ImportError: cannot import name 'BehaviourChangeWarning' from 'arviz.utils' (/Users/henryqu/Library/Caches/pypoetry/virtualenvs/nteprsm-kiQW9mOZ-py3.12/lib/python3.12/site-packages/arviz/utils.py)

In [ ]:
az.loo(azdata)

In [ ]:
az.loo(azdata_old)

## Seasonality

In [ ]:
# load model configuration
config_file = ROOT_DIR/"config/nteprsm_njkbg07.yml"
config = utils.load_config(config_file)
config["sampling"]['save_warmup'] = False
# process data
datahandler = utils.DataHandler(filepath='data/raw/quality_nj2.csv')
datahandler.model_data = df
datahandler.load_data()
datahandler.preprocess_data()
datahandler.generate_stan_data(**config["stan_additional_data"])

name2code = map_name2code(datahandler,'entry_name','entry_code', invert=True)
name2code

In [ ]:
"""colors = [
    "aliceblue", "antiquewhite", "aqua", "aquamarine", "azure",
    "beige", "bisque", "black", "blanchedalmond", "blue",
    "blueviolet", "brown", "burlywood", "cadetblue",
    "chartreuse", "chocolate", "coral", "cornflowerblue",
    "cornsilk", "crimson", "cyan", "darkblue", "darkcyan",
    "darkgoldenrod", "darkgray", "darkgrey", "darkgreen",
    "darkkhaki", "darkmagenta", "darkolivegreen", "darkorange",
    "darkorchid", "darkred", "darksalmon", "darkseagreen",
    "darkslateblue", "darkslategray", "darkslategrey",
    "darkturquoise", "darkviolet", "deeppink", "deepskyblue",
    "dimgray", "dimgrey", "dodgerblue", "firebrick",
    "floralwhite", "forestgreen", "fuchsia", "gainsboro",
    "ghostwhite", "gold", "goldenrod", "gray", "grey", "green",
    "greenyellow", "honeydew", "hotpink", "indianred", "indigo",
    "ivory", "khaki", "lavender", "lavenderblush", "lawngreen",
    "lemonchiffon", "lightblue", "lightcoral", "lightcyan",
    "lightgoldenrodyellow", "lightgray", "lightgrey",
    "lightgreen", "lightpink", "lightsalmon", "lightseagreen",
    "lightskyblue", "lightslategray", "lightslategrey",
    "lightsteelblue", "lightyellow", "lime", "limegreen",
    "linen", "magenta", "maroon", "mediumaquamarine",
    "mediumblue", "mediumorchid", "mediumpurple",
    "mediumseagreen", "mediumslateblue", "mediumspringgreen",
    "mediumturquoise", "mediumvioletred", "midnightblue",
    "mintcream", "mistyrose", "moccasin", "navajowhite", "navy",
    "oldlace", "olive", "olivedrab", "orange", "orangered",
    "orchid", "palegoldenrod", "palegreen", "paleturquoise",
    "palevioletred", "papayawhip", "peachpuff", "peru", "pink",
    "plum", "powderblue", "purple", "red", "rosybrown",
    "royalblue", "rebeccapurple", "saddlebrown", "salmon",
    "sandybrown", "seagreen", "seashell", "sienna", "silver",
    "skyblue", "slateblue", "slategray", "slategrey", "snow",
    "springgreen", "steelblue", "tan", "teal", "thistle", "tomato",
    "turquoise", "violet", "wheat", "white", "whitesmoke",
    "yellow", "yellowgreen"
]"""
colors = [
    "black", "blue", "blueviolet", "brown", "cadetblue",
    "chocolate", "coral", "crimson", "darkblue", "darkcyan",
    "darkgoldenrod", "darkgray", "darkgreen", "darkkhaki",
    "darkmagenta", "darkolivegreen", "darkorange", "darkorchid",
    "darkred", "darkseagreen", "darkslateblue", "darkslategray",
    "darkviolet", "deeppink", "deepskyblue", "dimgray",
    "dodgerblue", "firebrick", "forestgreen", "goldenrod",
    "gray", "green", "indianred", "indigo", "maroon",
    "mediumblue", "mediumorchid", "mediumpurple",
    "mediumseagreen", "mediumslateblue", "midnightblue",
    "navy", "olive", "olivedrab", "orangered", "purple",
    "red", "rosybrown", "royalblue", "saddlebrown", "seagreen",
    "sienna", "slateblue", "slategray", "teal", "tomato",
    "violet", "yellowgreen"
]
random.shuffle(colors)
entries = [i for i in range(89)]
random.shuffle(entries)
entries = entries[:10]

In [ ]:
colors = ['royalblue', 'mediumseagreen', 'deepskyblue', 'crimson']
entries = [6, 18, 28, 55]

In [ ]:
## params
plot_old_model = False
ci = 0.95  # Credible interval

# Create a single figure
fig = go.Figure()

# Define colors for each entry
#colors = [plt.cm.tab20(i) for i in range(20)]#['blue', 'green', 'red']

# time_effect_data_points
data_points = fit.time_effect
data_points_time = datahandler.model_data[datahandler.model_data['entry_code'] == 1]['adj_time_of_year']

for i, entry in enumerate(entries):
    entry_name = name2code[entry + 1]
    legend_group = f"Entry {entry_name}"    
    samples = fit.pred_time_effect[:, entry, :]  # Shape: (n_samples, n_time_points)
    
    # Compute statistics
    time_points = np.arange(samples.shape[1])+1  # Time indices
    mean_values = np.mean(samples, axis=0)  # Mean over samples
    lower_bound = np.percentile(samples, 100 * (1 - ci) / 2, axis=0)  # 2.5th percentile
    upper_bound = np.percentile(samples, 100 * (ci + (1 - ci) / 2), axis=0)  # 97.5th percentile

    # Add data point
    fig.add_trace(go.Scatter(
        x=data_points_time * 100, y=data_points.mean(axis=0)[entry],
        mode='markers',
        marker=dict(
            symbol='circle',  # Set the marker shape to "X"
            color=colors[i],  # Keep the color
            size=10  # Optionally, set the size of the marker
        ),
        name=f'{entry_name} Mean',
        legendgroup=legend_group,
        showlegend=False,
    ))

    # Add mean line
    fig.add_trace(go.Scatter(
        x=time_points, y=mean_values,
        mode='lines',
        line=dict(color=colors[i], width=2),
        name=f'{entry_name}',
        legendgroup=legend_group,
    ))
    
    # Add 95% Credible Interval (Shaded Region)
    fig.add_trace(go.Scatter(
        x=time_points.tolist() + time_points[::-1].tolist(),
        y=upper_bound.tolist() + lower_bound[::-1].tolist(),
        fill='toself',
        fillcolor=hex_to_rgba(colors[i], alpha=0.2)  ,
        line=dict(color='rgba(255,255,255,0)'),
        name=f'{entry_name} 95% CI',
        legendgroup=legend_group,  # Grouping
        showlegend=False,
    ))

    # Add old model horizontal baseline as a scatter trace (instead of add_shape)
    if plot_old_model:
        old_model_samples = fit_old_model.entry[:, entry]
    
        entry_old_model = old_model_samples.mean()
        fig.add_trace(go.Scatter(
            x=[time_points[0], time_points[-1]],  # Span the entire x-axis
            y=[entry_old_model, entry_old_model],  # Constant y-value
            mode='lines',
            line=dict(color=colors[i], width=2, dash="dash"),
            name=f'{entry_name} Baseline',
            legendgroup=legend_group,  # Grouping
            showlegend=False,  # No separate legend entry
            visible=True
        ))

        lower_bound_old_model = np.percentile(old_model_samples, 100 * (1 - ci) / 2, axis=0)  # 2.5th percentile
        upper_bound_old_model = np.percentile(old_model_samples, 100 * (ci + (1 - ci) / 2), axis=0)  # 97.5th percentile
    
        # Add 95% Credible Interval (Shaded Region)
        fig.add_trace(go.Scatter(
            x=time_points.tolist() + time_points[::-1].tolist(),
            y=[lower_bound_old_model]*len(time_points) + [upper_bound_old_model]*len(time_points),
            fill='toself',
            fillcolor=hex_to_rgba(colors[i], alpha=0.2)  ,
            line=dict(color='rgba(255,255,255,0)'),
            name=f'Entry {entry} 95% CI',
            legendgroup=legend_group,  # Grouping
            showlegend=False,
        ))

# Compute correct tick positions based on actual day of the year
days_in_month = np.array([0, 31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31])  # Non-leap year
cumulative_days = np.cumsum(days_in_month)+1  # Cumulative sum to get end of each month
month_positions = 100*cumulative_days / 365  # Normalize to range 0-100
month_positions[0] += 0.8
#month_positions = np.insert(month_positions, 0, 0)
# Generate month labels
month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

fig.update_layout(
    xaxis=dict(
        tickvals=month_positions,  # Correctly spaced tick positions
        ticktext=month_labels,  # Month names
        title="Time of Year"
    ),
    yaxis_title="Seasonality (Latent Scale)",
    title=f"Turfgrass Seasonality, {ci} credible interval",
    template="ggplot2",
    legend=dict(title="Entries")
)


# Show interactive plot
fig.show()

In [ ]:
fit.tau_rater.mean(axis=0).shape

In [ ]:
rater_mapping = ['A', 'B', 'C', 'D', 'E', 'F', 'G']
for i in range(7):
    fig = plot_rater_characteristic_curve(fit.tau_rater.mean(axis=0)[i,:], colors=NTEP_COLOR_SCALE)
    fig.update_layout(
        title=f"Category probabilities for rater {rater_mapping[i]}",
        template="plotly_white"
    )
    fig.show()

In [ ]:
fit.tau_rater.mean(axis=0)[0,:]

In [ ]:
# Build the figure
thresholds = [fit.tau_rater.mean(axis=0)[i,:] for i in range(7)]
x_start, x_end = -6, 6
fig = go.Figure()

for i, (rater, rater_thresholds) in enumerate(zip(rater_mapping, thresholds)):
    # Add the start and end points for categories
    #fake_thresholds = [x_start] + list(rater_thresholds) + [x_end]
    rater_thresholds = [x_start] + list(rater_thresholds) + [x_end]
    
    for j in range(len(rater_thresholds) - 1):
        # Add a rectangle for each category
        fig.add_trace(go.Bar(
            x=[rater_thresholds[j + 1] - rater_thresholds[j]],  # Width of the bar
            y=[rater],
            orientation='h',
            marker=dict(color=NTEP_COLOR_SCALE[j % len(NTEP_COLOR_SCALE)]),
            name=f"{j + 1}" if i == 0 else None,  # Show legend only for the first rater
            showlegend=(i == 0),  # Show legend only once
            base=rater_thresholds[j]
        ))

# Update layout
fig.update_layout(
    barmode="stack",
    title="Most Probable Rating Outcome",
    xaxis=dict(title="Turf Quality on Latent Scale", range=[x_start-0.05, x_end+0.05]),
    yaxis=dict(title="Rater", categoryorder="category ascending"),
    showlegend=True
)

fig.update_yaxes(autorange="reversed")

# Show the plot
fig.show()